# 09 — So sánh GRU và Transformer trên Test

Test (09/2025–08/2026) chỉ được dùng ở đây, một lần, sau khi 07 và 08 đã chốt mô hình.
Mọi phương pháp được chấm trên cùng các thời điểm phát hành dự báo và cùng horizon 3/6/12/24h.

In [ ]:
# Chạy trên Colab (GPU): bỏ comment, sửa link repo của bạn
# !git clone <link-repo> repo
# %cd repo/notebooks

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import forecasting as fc

fc.enable_determinism()
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
MODELS = {'GRU': 'gru', 'Transformer': 'transformer'}
REPORT_DIR = ROOT / 'results' / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

finals = {name: json.loads((ROOT / 'results' / 'models' / d / 'final_config.json').read_text(encoding='utf-8'))
          for name, d in MODELS.items()}
assert finals['GRU']['features'] == finals['Transformer']['features'], 'Hai mô hình phải dùng cùng feature'
data = fc.WindowData(ROOT, finals['GRU']['features'])

pos = data.positions['Test']
y_test = data.targets(pos)
preds = {}
for name, d in MODELS.items():
    long = pd.read_csv(ROOT / 'results' / 'models' / d / 'predictions_test.csv', parse_dates=['issue_time'])
    wide = long.pivot(index='issue_time', columns='horizon', values='y_pred')[data.H.tolist()]
    assert wide.index.equals(pd.DatetimeIndex(data.index[pos])), 'Thời điểm dự báo không khớp'
    preds[name] = wide.to_numpy()
preds.update(fc.baselines(data, 'Test'))
print('Số mẫu Test:', len(pos), '|', data.index[pos[0]], '→', data.index[pos[-1]])

## I. Bảng kết quả chính

In [ ]:
rows = []
for method, p in preds.items():
    t = fc.metrics_table(y_test, p, data)
    t.insert(0, 'Method', method)
    rows.append(t)
test_table = pd.concat(rows, ignore_index=True)
test_table.to_csv(REPORT_DIR / 'test_metrics.csv', index=False)

order = list(preds)
cols = [f'{h}h' for h in data.H] + ['All']
for metric in ['MAE', 'RMSE', 'R2', 'High_PM_MAE']:
    print(metric)
    display(test_table.pivot(index='Method', columns='Horizon', values=metric).loc[order, cols].round(2))

In [ ]:
# Skill so với persistence: 1 - MAE(mô hình) / MAE(persistence). Dương = tốt hơn persistence.
mae_tab = test_table.pivot(index='Method', columns='Horizon', values='MAE').loc[order, cols]
skill = (1 - mae_tab / mae_tab.loc['Persistence']) * 100
skill.to_csv(REPORT_DIR / 'test_skill_vs_persistence.csv')
display(skill.round(1).astype(str) + '%')

## II. Khác biệt có ý nghĩa thống kê không? (Diebold–Mariano)

So sai số tuyệt đối của hai phương pháp trên cùng mẫu. Thống kê âm nghĩa là phương pháp thứ nhất tốt hơn;
p < 0,05 thì khác biệt có ý nghĩa.

In [ ]:
pairs = [('GRU', 'Transformer'), ('GRU', 'Persistence'), ('Transformer', 'Persistence')]
dm_rows = []
for a, b in pairs:
    for j, h in enumerate(data.H):
        stat, p = fc.diebold_mariano(y_test[:, j] - preds[a][:, j], y_test[:, j] - preds[b][:, j], int(h))
        dm_rows.append({'A': a, 'B': b, 'Horizon': f'{h}h', 'DM_stat': stat, 'p_value': p,
                        'Kết luận': (f'{a} tốt hơn' if stat < 0 else f'{b} tốt hơn') if p < 0.05 else 'Không khác biệt rõ'})
dm = pd.DataFrame(dm_rows)
dm.to_csv(REPORT_DIR / 'test_diebold_mariano.csv', index=False)
display(dm.round(4))

## III. Biểu đồ

In [ ]:
x = np.arange(len(data.H))
width = 0.8 / len(order)
fig, ax = plt.subplots(figsize=(8, 4))
for i, m in enumerate(order):
    ax.bar(x + i * width, mae_tab.loc[m, [f'{h}h' for h in data.H]], width, label=m)
ax.set_xticks(x + width * (len(order) - 1) / 2, [f'{h}h' for h in data.H])
ax.set_xlabel('Horizon'); ax.set_ylabel('MAE (µg/m³)'); ax.set_title('MAE trên Test theo horizon')
ax.legend(); plt.tight_layout()
plt.savefig(REPORT_DIR / 'mae_by_horizon.png', dpi=200); plt.show()

In [ ]:
# Một tháng mùa đông để xem mô hình bám đỉnh ô nhiễm thế nào (target_time = issue_time + h)
SHOW_START, SHOW_END = '2025-12-01', '2025-12-31'
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, h in zip(axes, [6, 24]):
    j = data.H.tolist().index(h)
    t = data.index[pos + h]
    mask = (t >= SHOW_START) & (t <= SHOW_END)
    ax.plot(t[mask], y_test[mask, j], color='black', lw=1.2, label='Thực tế')
    for m in ['GRU', 'Transformer']:
        ax.plot(t[mask], preds[m][mask, j], lw=1, label=m)
    ax.set_title(f'Dự báo trước {h} giờ'); ax.set_ylabel('PM2.5 (µg/m³)'); ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(REPORT_DIR / 'timeseries_winter.png', dpi=200); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
j = data.H.tolist().index(24)
lim = float(np.percentile(y_test[:, j], 99.5))
for ax, m in zip(axes, ['GRU', 'Transformer']):
    ax.scatter(y_test[:, j], preds[m][:, j], s=2, alpha=0.3)
    ax.plot([0, lim], [0, lim], 'k--', lw=1)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(f'{m} — dự báo 24h'); ax.set_xlabel('Thực tế (µg/m³)')
axes[0].set_ylabel('Dự báo (µg/m³)')
plt.tight_layout(); plt.savefig(REPORT_DIR / 'scatter_24h.png', dpi=200); plt.show()

In [ ]:
# MAE theo tháng (trung bình 4 horizon): mô hình yếu ở mùa nào?
month = pd.Series(data.index[pos]).dt.to_period('M').astype(str).to_numpy()
monthly = pd.DataFrame({m: pd.Series(np.abs(y_test - preds[m]).mean(axis=1)).groupby(month).mean()
                        for m in order})
monthly.to_csv(REPORT_DIR / 'test_mae_by_month.csv')
monthly.plot(marker='o', figsize=(11, 4), ylabel='MAE (µg/m³)', title='MAE theo tháng trên Test')
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig(REPORT_DIR / 'mae_by_month.png', dpi=200); plt.show()

## IV. Cấu hình và chi phí của hai mô hình

In [ ]:
info_rows = []
for name, d in MODELS.items():
    tuning = pd.read_csv(ROOT / 'results' / 'models' / d / 'tuning.csv').sort_values('val_MAE')
    best = tuning.iloc[0]
    info_rows.append({'Model': name, **finals[name]['config'], 'n_params': int(best['n_params']),
                      'best_epoch': int(best['best_epoch']), 'train_seconds_1seed': best['train_seconds'],
                      'n_trials': len(tuning), 'val_MAE_best_trial': best['val_MAE']})
model_info = pd.DataFrame(info_rows).set_index('Model').T
model_info.to_csv(REPORT_DIR / 'model_configs.csv')
display(model_info)
print('Feature dùng chung:', finals['GRU']['features'])
print('Đã lưu bảng và hình vào', REPORT_DIR)